In [52]:
# pip install gspread google-auth pandas

In [64]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import gspread

from google.oauth2.service_account import Credentials
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:

# CONFIGURACIÓN

# Este script/notebook está en 3.Modelo_ML/Modelos_semanales/
DATA_DIR = "../../Datos_csv"

# El json está en 3.Modelo_ML/Excel_experimentos/
SERVICE_ACCOUNT_FILE = "credenciales_google.json"

SHEET_URL = "https://docs.google.com/spreadsheets/d/17T0L6gza7vzD1vFh6OG2j3qOmlKCDRtFFu5HNma1Agk/edit?gid=836870544#gid=836870544"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]


# CONEXIÓN GOOGLE SHEETS

def conectar_sheet():
    creds = Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=SCOPES
    )
    client = gspread.authorize(creds)
    spreadsheet = client.open_by_url(SHEET_URL)
    worksheet = spreadsheet.get_worksheet(0)
    return worksheet


def leer_experimentos(worksheet):
    records = worksheet.get_all_records()
    df = pd.DataFrame(records)
    return df

def buscar_primer_bloque_linear_weekly(df):
    for idx, row in df.iterrows():
        estado = str(row.get("Estado", "")).strip().upper()
        archivo = str(row.get("Archivo", ""))
        metodo = str(row.get("Método", ""))

        if estado != "PENDIENTE":
            continue

        if metodo != "LinearRegression":
            continue

        if not archivo.startswith("df_semanal"):
            continue

        fila_sheet = idx + 2
        return fila_sheet, row.to_dict(), idx

    return None, None, None


def buscar_primer_bloque_ridge_weekly(df):
    for idx, row in df.iterrows():
        estado = str(row.get("Estado", "")).strip().upper()
        archivo = str(row.get("Archivo", ""))
        metodo = str(row.get("Método", ""))

        if estado != "PENDIENTE":
            continue

        if metodo != "Ridge":
            continue

        if not archivo.startswith("df_semanal"):
            continue

        fila_sheet = idx + 2
        return fila_sheet, row.to_dict(), idx

    return None, None, None


In [ ]:
def get_model(nombre_metodo):
    if nombre_metodo == "LinearRegression":
        return LinearRegression()

    elif nombre_metodo == "Ridge":
        return Ridge(alpha=1.0)

    else:
        raise ValueError(f"No se permite el método: {nombre_metodo}")

In [56]:

def obtener_filas_bloque_pendientes(df_exp, archivo, metodo, idx_inicio):
    filas = []

    for idx in range(idx_inicio, len(df_exp)):
        row = df_exp.iloc[idx]

        archivo_row = row["Archivo"]
        metodo_row = row["Método"]
        estado_row = str(row.get("Estado", "")).strip().upper()

        if archivo_row != archivo or metodo_row != metodo:
            break

        if estado_row == "PENDIENTE":
            filas.append({
                "idx_df": idx,
                "fila_sheet": idx + 2,
                "target": row["Target"]
            })

    return filas

In [57]:

def marcar_bloque_como_running(worksheet, filas_bloque):
    fecha_ini = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    fila_ini = filas_bloque[0]["fila_sheet"]
    fila_fin = filas_bloque[-1]["fila_sheet"]

    values = []
    for _ in filas_bloque:
        values.append([fecha_ini, "", "", "", "", "", "", "", "RUNNING"])

    worksheet.update(
        range_name=f"D{fila_ini}:L{fila_fin}",
        values=values
    )

    return fecha_ini


def marcar_bloque_como_done(worksheet, filas_bloque, resultados_metricas):
    fecha_fin = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    fila_ini = filas_bloque[0]["fila_sheet"]
    fila_fin = filas_bloque[-1]["fila_sheet"]

    values = []
    for res in resultados_metricas:
        values.append([
            fecha_fin,
            round(res["rmse"], 5),
            round(res["rmse_baseline"], 5),
            round(res["rmse_zeros"], 5),
            round(res["mae"], 5),
            round(res["mae_baseline"], 5),
            round(res["mae_zeros"], 5),
            "DONE"
        ])

    worksheet.update(
        range_name=f"E{fila_ini}:L{fila_fin}",
        values=values
    )

    return fecha_fin


def marcar_bloque_como_error(worksheet, filas_bloque):
    fecha_fin = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    fila_ini = filas_bloque[0]["fila_sheet"]
    fila_fin = filas_bloque[-1]["fila_sheet"]

    values = []
    for _ in filas_bloque:
        values.append([fecha_fin, "", "", "", "", "", "", "ERROR"])

    worksheet.update(
        range_name=f"E{fila_ini}:L{fila_fin}",
        values=values
    )

    return fecha_fin

In [58]:
def preprocess_weekly_data(df, target_actual):
    df = df.copy()

    if target_actual not in df.columns:
        raise ValueError(f"El target {target_actual} no existe en el dataframe.")

    if "week_key" not in df.columns:
        raise ValueError("La columna 'week_key' no existe en el dataframe.")

    week_key = df["week_key"].copy()
    y = df[target_actual].copy()

    columnas_drop = ["week_key", "year", "semana", "mes"]
    df = df.drop(columns=columnas_drop, errors="ignore")

    target_cols = [c for c in df.columns if c.startswith("target_")]
    X = df.drop(columns=target_cols, errors="ignore")

    X = X.select_dtypes(include=[np.number])

    mask_valida = pd.concat([X, y.rename(target_actual)], axis=1).notna().all(axis=1)

    X = X.loc[mask_valida].reset_index(drop=True)
    y = y.loc[mask_valida].reset_index(drop=True)
    week_key = week_key.loc[mask_valida].reset_index(drop=True)

    return X, y, week_key

In [59]:

def temporal_split_70_30(X, y, week_key, train_ratio=0.7):
    n = len(X)
    split_idx = int(n * train_ratio)

    if n < 10:
        raise ValueError(f"Muy pocas observaciones para hacer split: n={n}")

    if split_idx <= 0 or split_idx >= n:
        raise ValueError(f"Split inválido: n={n}, split_idx={split_idx}")

    X_train = X.iloc[:split_idx].copy()
    X_test = X.iloc[split_idx:].copy()

    y_train = y.iloc[:split_idx].copy()
    y_test = y.iloc[split_idx:].copy()

    info_split = {
        "n_total": n,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "train_week_ini": str(week_key.iloc[0]),
        "train_week_fin": str(week_key.iloc[split_idx - 1]),
        "test_week_ini": str(week_key.iloc[split_idx]),
        "test_week_fin": str(week_key.iloc[len(week_key) - 1]),
    }

    return X_train, X_test, y_train, y_test, info_split

In [ ]:
def walk_forward_regression(X_train, y_train, X_test, y_test, metodo):
    preds_model = []
    preds_baseline = []
    preds_zeros = []

    X_hist = X_train.copy().reset_index(drop=True)
    y_hist = y_train.copy().reset_index(drop=True)

    X_test = X_test.copy().reset_index(drop=True)
    y_test = y_test.copy().reset_index(drop=True)

    for i in range(len(X_test)):
        model = get_model(metodo)
        model.fit(X_hist, y_hist)

        x_next = X_test.iloc[[i]]

        pred_model = model.predict(x_next)[0]
        preds_model.append(pred_model)

        pred_baseline = y_hist.iloc[-1]
        preds_baseline.append(pred_baseline)

        preds_zeros.append(0.0)

        X_hist = pd.concat([X_hist, x_next], ignore_index=True)
        y_hist = pd.concat(
            [y_hist, pd.Series([y_test.iloc[i]])],
            ignore_index=True
        )

    rmse = np.sqrt(mean_squared_error(y_test, preds_model))
    rmse_baseline = np.sqrt(mean_squared_error(y_test, preds_baseline))
    rmse_zeros = np.sqrt(mean_squared_error(y_test, preds_zeros))

    mae = mean_absolute_error(y_test, preds_model)
    mae_baseline = mean_absolute_error(y_test, preds_baseline)
    mae_zeros = mean_absolute_error(y_test, preds_zeros)

    resultados = pd.DataFrame({
        "real": y_test,
        "pred_model": preds_model,
        "pred_baseline": preds_baseline,
        "pred_zeros": preds_zeros
    })

    return (
        rmse,
        rmse_baseline,
        rmse_zeros,
        mae,
        mae_baseline,
        mae_zeros,
        resultados
    )

In [ ]:

def run_block_linear_weekly():
    worksheet = conectar_sheet()
    df_exp = leer_experimentos(worksheet)

    fila_sheet, experimento, idx_inicio = buscar_primer_bloque_linear_weekly(df_exp)
    if fila_sheet is None:
        print("No hay experimentos pendientes.")
        return None

    archivo = experimento["Archivo"]
    metodo = experimento["Método"]

    print("=====================================================")
    print(f"INICIO DE BLOQUE : Archivo ->{archivo}   Método -> {metodo}")

    path = os.path.join(DATA_DIR, archivo)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el archivo: {path}")

    df_cargado = pd.read_csv(path)

    filas_bloque = obtener_filas_bloque_pendientes(df_exp, archivo, metodo, idx_inicio)

    if not filas_bloque:
        print("No hay filas pendientes válidas en el bloque.")
        return None

    target_inicial = filas_bloque[0]["target"]
    X0, y0, week_key0 = preprocess_weekly_data(df_cargado, target_inicial)
    _, _, _, _, info_split = temporal_split_70_30(
        X=X0,
        y=y0,
        week_key=week_key0,
        train_ratio=0.7
    )

    print("INFORMACIÓN DEL SPLIT TEMPORAL DEL BLOQUE")
    print(f"Total observaciones: {info_split['n_total']}")
    print(f"Train: {info_split['n_train']} filas")
    print(f"Test : {info_split['n_test']} filas")
    print(f"Train semanas: {info_split['train_week_ini']} -> {info_split['train_week_fin']}")
    print(f"Test semanas : {info_split['test_week_ini']} -> {info_split['test_week_fin']}")

    fecha_ini = marcar_bloque_como_running(worksheet, filas_bloque)

    resultados_bloque = {}
    resultados_metricas = []

    try:
        for fila in filas_bloque:
            target = fila["target"]

            X, y, week_key = preprocess_weekly_data(df_cargado, target)

            X_train, X_test, y_train, y_test, _ = temporal_split_70_30(
                X=X,
                y=y,
                week_key=week_key,
                train_ratio=0.7
            )

            (
                rmse,
                rmse_baseline,
                rmse_zeros,
                mae,
                mae_baseline,
                mae_zeros,
                resultados
            ) = walk_forward_regression(X_train, y_train, X_test, y_test)

            resultados_bloque[target] = resultados

            resultados_metricas.append({
                "rmse": rmse,
                "rmse_baseline": rmse_baseline,
                "rmse_zeros": rmse_zeros,
                "mae": mae,
                "mae_baseline": mae_baseline,
                "mae_zeros": mae_zeros
            })

            print(
                f"Target: {target} | "
                f"RMSE={round(rmse, 5)} | "
                f"RMSE_base={round(rmse_baseline, 5)} | "
                f"RMSE_0={round(rmse_zeros, 5)} | "
                f"MAE={round(mae, 5)} | "
                f"MAE_base={round(mae_baseline, 5)} | "
                f"MAE_0={round(mae_zeros, 5)}"
            )

        fecha_fin = marcar_bloque_como_done(worksheet, filas_bloque, resultados_metricas)

        print("FIN DE BLOQUE")
        print(f"Fecha inicio bloque: {fecha_ini}")
        print(f"Fecha fin bloque   : {fecha_fin}")
        print(f"Targets ejecutados : {len(resultados_bloque)}")
        print("=====================================================")

        return resultados_bloque, info_split

    except Exception as e:
        marcar_bloque_como_error(worksheet, filas_bloque)
        print("ERROR en el bloque:")
        print(str(e))
        raise

In [ ]:
def run_block_ridge_weekly():
    worksheet = conectar_sheet()
    df_exp = leer_experimentos(worksheet)

    fila_sheet, experimento, idx_inicio = buscar_primer_bloque_ridge_weekly(df_exp)

    if fila_sheet is None:
        print("No quedan bloques semanales pendientes para Ridge.")
        return None

    archivo = experimento["Archivo"]
    metodo = experimento["Método"]

    print("=====================================================")
    print("INICIO DE BLOQUE")
    print(f"Archivo: {archivo}")
    print(f"Método: {metodo}")

    path = os.path.join(DATA_DIR, archivo)
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el archivo: {path}")

    df_cargado = pd.read_csv(path)

    filas_bloque = obtener_filas_bloque_pendientes(df_exp, archivo, metodo, idx_inicio)

    if not filas_bloque:
        print("No hay filas pendientes válidas en el bloque.")
        return None

    target_inicial = filas_bloque[0]["target"]
    X0, y0, week_key0 = preprocess_weekly_data(df_cargado, target_inicial)
    _, _, _, _, info_split = temporal_split_70_30(
        X=X0,
        y=y0,
        week_key=week_key0,
        train_ratio=0.7
    )

    print("INFORMACIÓN DEL SPLIT TEMPORAL DEL BLOQUE")
    print(f"Total observaciones: {info_split['n_total']}")
    print(f"Train: {info_split['n_train']} filas")
    print(f"Test : {info_split['n_test']} filas")
    print(f"Train semanas: {info_split['train_week_ini']} -> {info_split['train_week_fin']}")
    print(f"Test semanas : {info_split['test_week_ini']} -> {info_split['test_week_fin']}")

    fecha_ini = marcar_bloque_como_running(worksheet, filas_bloque)

    resultados_bloque = {}
    resultados_metricas = []

    try:
        for fila in filas_bloque:
            target = fila["target"]

            X, y, week_key = preprocess_weekly_data(df_cargado, target)

            X_train, X_test, y_train, y_test, _ = temporal_split_70_30(
                X=X,
                y=y,
                week_key=week_key,
                train_ratio=0.7
            )

            (
                rmse,
                rmse_baseline,
                rmse_zeros,
                mae,
                mae_baseline,
                mae_zeros,
                resultados
            ) = walk_forward_regression(
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                metodo="Ridge"
            )

            resultados_bloque[target] = resultados

            resultados_metricas.append({
                "rmse": rmse,
                "rmse_baseline": rmse_baseline,
                "rmse_zeros": rmse_zeros,
                "mae": mae,
                "mae_baseline": mae_baseline,
                "mae_zeros": mae_zeros
            })

            print(
                f"Target: {target} | "
                f"RMSE={round(rmse, 5)} | "
                f"RMSE_base={round(rmse_baseline, 5)} | "
                f"RMSE_0={round(rmse_zeros, 5)} | "
                f"MAE={round(mae, 5)} | "
                f"MAE_base={round(mae_baseline, 5)} | "
                f"MAE_0={round(mae_zeros, 5)}"
            )

        fecha_fin = marcar_bloque_como_done(worksheet, filas_bloque, resultados_metricas)

        print("FIN DE BLOQUE")
        print(f"Fecha inicio bloque: {fecha_ini}")
        print(f"Fecha fin bloque   : {fecha_fin}")
        print(f"Targets ejecutados : {len(resultados_bloque)}")

        return resultados_bloque, info_split

    except Exception as e:
        marcar_bloque_como_error(worksheet, filas_bloque)
        print("ERROR en el bloque:")
        print(str(e))
        raise

Ejecutar todos los linear regression

In [62]:
while True:
    out = run_block_linear_weekly()
    if out is None:
        break

INICIO DE BLOQUE : Archivo ->df_semanal_1.csv   Método -> LinearRegression
INFORMACIÓN DEL SPLIT TEMPORAL DEL BLOQUE
Total observaciones: 264
Train: 184 filas
Test : 80 filas
Train semanas: 2021-W02 -> 2024-W29
Test semanas : 2024-W30 -> 2026-W05
Target: target_AGG | RMSE=0.00468 | RMSE_base=0.00226 | RMSE_0=0.00153 | MAE=0.00335 | MAE_base=0.00169 | MAE_0=0.00113
Target: target_BND | RMSE=0.0047 | RMSE_base=0.00221 | RMSE_0=0.00151 | MAE=0.00335 | MAE_base=0.00165 | MAE_0=0.00111
Target: target_DBC | RMSE=0.01611 | RMSE_base=0.00807 | RMSE_0=0.00614 | MAE=0.01184 | MAE_base=0.00587 | MAE_0=0.00394
Target: target_DIA | RMSE=0.01267 | RMSE_base=0.00683 | RMSE_0=0.00447 | MAE=0.0089 | MAE_base=0.00523 | MAE_0=0.00349
Target: target_DVY | RMSE=0.01415 | RMSE_base=0.00601 | RMSE_0=0.00402 | MAE=0.01055 | MAE_base=0.00441 | MAE_0=0.00299
Target: target_EEM | RMSE=0.01307 | RMSE_base=0.00676 | RMSE_0=0.00476 | MAE=0.01046 | MAE_base=0.00544 | MAE_0=0.00379
Target: target_EFA | RMSE=0.01339 |

Ejecutar todos los Ridge Regression

In [ ]:
while True:
    out = run_block_ridge_weekly()
    if out is None:
        print("No quedan bloques semanales de Ridge pendientes.")
        break

In [63]:
# df = pd.read_csv("../../Datos_csv/df_semanal_2.csv")
# pd.set_option("display.max_columns", None) 
# df.head()